# QP-001: Quantum Processor Simulator

Classical circuit simulations for learning with Qiskit, Aer, and PennyLane. Run cells from top to bottom. This notebook does not access quantum hardware.


In [ ]:
# In Jupyter/Colab, install into the active kernel. For local setup, prefer: python -m pip install -r requirements.txt
%pip install --quiet 'qiskit>=1.0,<3.0' 'qiskit-aer>=0.15,<0.19' 'pennylane>=0.40,<0.46' 'numpy>=1.23,<3.0' 'matplotlib>=3.7,<4.0'

import numpy as np
import matplotlib.pyplot as plt
import qiskit
import qiskit_aer
import pennylane as qml
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print('Qiskit:', qiskit.__version__)
print('Qiskit Aer:', qiskit_aer.__version__)
print('PennyLane:', qml.__version__)


## 1. Bell state (Qiskit Aer)

The ideal measurement outcomes are `00` and `11` with equal probability.


In [ ]:
bell = QuantumCircuit(2, 2)
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])
print(bell.draw(output='text'))

shots = 1024
bell_counts = AerSimulator().run(bell, shots=shots, seed_simulator=42).result().get_counts()
print('Bell counts:', bell_counts)

bell_labels = sorted(bell_counts)
bell_colors = {'00': '#4C72B0', '11': '#55A868'}
plt.figure(figsize=(6, 4))
plt.bar(bell_labels, [bell_counts[label] for label in bell_labels], color=[bell_colors.get(label, '#4C72B0') for label in bell_labels])
plt.title('Bell-state measurement distribution')
plt.xlabel('Outcome')
plt.ylabel('Counts')
plt.tight_layout()
plt.show()


## 2. GHZ state (Qiskit Aer)

An ideal `n`-qubit GHZ circuit has only all-zero and all-one computational-basis outcomes.


In [ ]:
def ghz_circuit(n_qubits=8):
    if isinstance(n_qubits, bool) or not isinstance(n_qubits, int) or n_qubits < 1:
        raise ValueError('n_qubits must be a positive integer')
    circuit = QuantumCircuit(n_qubits, n_qubits)
    circuit.h(0)
    for qubit in range(n_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure(range(n_qubits), range(n_qubits))
    return circuit

n_qubits = 8
ghz_counts = AerSimulator().run(ghz_circuit(n_qubits), shots=4096, seed_simulator=42).result().get_counts()
significant_counts = {state: count for state, count in ghz_counts.items() if count > 10}
print('Significant GHZ outcomes:', significant_counts)

plt.figure(figsize=(9, 4))
plt.bar(list(significant_counts), list(significant_counts.values()), color='#DD8452')
plt.title(f'GHZ measurement outcomes (n={n_qubits})')
plt.xlabel('Bitstring')
plt.ylabel('Counts')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 3. Differentiable PennyLane circuit


In [ ]:
gradient_device = qml.device('default.qubit', wires=1)

@qml.qnode(gradient_device, shots=None)
def parameterized_circuit(phi1, phi2):
    qml.RX(phi1, wires=0)
    qml.RY(phi2, wires=0)
    return qml.expval(qml.PauliZ(0))

def cost(phi1, phi2):
    return qml.numpy.sin(qml.numpy.abs(parameterized_circuit(phi1, phi2))) - 1

phi1 = qml.numpy.array(0.5, requires_grad=True)
phi2 = qml.numpy.array(0.5, requires_grad=True)
gradient = qml.grad(cost, argnum=[0, 1])(phi1, phi2)
print('Gradient:', gradient)


## 4. Noisy GHZ experiment

The all-zero/all-one percentage is a computational-basis population score, not a full state-fidelity measurement.


In [ ]:
n_qubits = 8
noise_level = 0.01
shots = 2000
if not 0.0 <= noise_level <= 1.0:
    raise ValueError('noise_level must be between 0 and 1')

noisy_device = qml.device('default.mixed', wires=n_qubits)

@qml.qnode(noisy_device, shots=shots)
def noisy_ghz():
    qml.Hadamard(wires=0)
    for qubit in range(n_qubits - 1):
        qml.CNOT(wires=[qubit, qubit + 1])
        qml.DepolarizingChannel(noise_level, wires=qubit)
        qml.DepolarizingChannel(noise_level, wires=qubit + 1)
    return qml.sample(wires=range(n_qubits))

samples = np.asarray(noisy_ghz(), dtype=int).reshape(shots, n_qubits)
states, counts = np.unique(samples, axis=0, return_counts=True)
measurement_counts = {''.join(map(str, state)): int(count) for state, count in zip(states, counts)}
zero_count = measurement_counts.get('0' * n_qubits, 0)
one_count = measurement_counts.get('1' * n_qubits, 0)
ghz_population = 100.0 * (zero_count + one_count) / shots
print('Noise level:', noise_level)
print('GHZ-basis population:', round(ghz_population, 2), '%')
print('Most frequent outcomes:', sorted(measurement_counts.items(), key=lambda item: item[1], reverse=True)[:10])


## 5. Three-qubit bit-flip repetition-code demonstration

This encodes a known logical `|1⟩`, applies independent bit flips, and compares one physical bit with a majority-vote decoded bit. It is not a complete quantum error-correcting code.


In [ ]:
def repetition_code_experiment(error_rate=0.05, shots=5000):
    if isinstance(error_rate, bool) or not isinstance(error_rate, (int, float)) or not 0.0 <= error_rate <= 1.0:
        raise ValueError('error_rate must be a number between 0 and 1')
    if isinstance(shots, bool) or not isinstance(shots, int) or shots <= 0:
        raise ValueError('shots must be a positive integer')
    device = qml.device('default.mixed', wires=3)

    @qml.qnode(device, shots=shots)
    def encoded_state_with_bit_flips():
        qml.PauliX(wires=0)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[0, 2])
        for wire in range(3):
            qml.BitFlip(error_rate, wires=wire)
        return qml.sample(wires=[0, 1, 2])

    results = np.asarray(encoded_state_with_bit_flips(), dtype=int).reshape(shots, 3)
    raw_errors = int(np.count_nonzero(results[:, 0] != 1))
    decoded_bits = (results.sum(axis=1) >= 2).astype(int)
    decoded_errors = int(np.count_nonzero(decoded_bits != 1))
    return {'raw_errors': raw_errors, 'decoded_errors': decoded_errors, 'raw_error_rate': raw_errors / shots, 'decoded_error_rate': decoded_errors / shots}

code_results = repetition_code_experiment()
print('Physical qubit-0 error rate:', f"{code_results['raw_error_rate']:.2%}")
print('Majority-decoded logical error rate:', f"{code_results['decoded_error_rate']:.2%}")


## 6. Reusable PennyLane processor simulator


In [ ]:
class QuantumProcessorSimulator:
    def __init__(self, qubits=4, noise_level=0.0):
        if isinstance(qubits, bool) or not isinstance(qubits, int) or qubits < 2:
            raise ValueError('qubits must be an integer of at least 2')
        if isinstance(noise_level, bool) or not isinstance(noise_level, (int, float)) or not 0.0 <= noise_level <= 1.0:
            raise ValueError('noise_level must be a number between 0 and 1')
        self.qubits = qubits
        self.noise_level = float(noise_level)
        device_name = 'default.mixed' if self.noise_level > 0 else 'default.qubit'
        self.device = qml.device(device_name, wires=qubits)

    def _apply_noise(self, wires):
        if self.noise_level > 0:
            for wire in wires:
                qml.DepolarizingChannel(self.noise_level, wires=wire)

    @staticmethod
    def _validate_shots(shots):
        if isinstance(shots, bool) or not isinstance(shots, int) or shots <= 0:
            raise ValueError('shots must be a positive integer')

    def bell_state(self, shots=1024):
        self._validate_shots(shots)
        @qml.qnode(self.device, shots=shots)
        def circuit():
            qml.Hadamard(wires=0)
            self._apply_noise([0])
            qml.CNOT(wires=[0, 1])
            self._apply_noise([0, 1])
            return qml.sample(wires=[0, 1])
        return circuit()

    def ghz_state(self, shots=1024):
        self._validate_shots(shots)
        @qml.qnode(self.device, shots=shots)
        def circuit():
            qml.Hadamard(wires=0)
            self._apply_noise([0])
            for qubit in range(self.qubits - 1):
                qml.CNOT(wires=[qubit, qubit + 1])
                self._apply_noise([qubit, qubit + 1])
            return qml.sample(wires=range(self.qubits))
        return circuit()

    def summary(self):
        return {'qubits': self.qubits, 'noise_level': self.noise_level, 'device': 'default.mixed' if self.noise_level > 0 else 'default.qubit'}

processor = QuantumProcessorSimulator(qubits=4, noise_level=0.0)
print('Processor:', processor.summary())
print('Bell samples:', processor.bell_state(shots=100)[:10])
print('GHZ samples:', processor.ghz_state(shots=20)[:10])


## Interpretation and limitations

- The simulator runs locally and is not connected to quantum hardware.
- Results are finite-shot estimates; most measurements are stochastic.
- Noise is illustrative. The GHZ population is not full state fidelity.
- The repetition-code example only demonstrates majority decoding for known computational-basis data under bit-flip noise.
- Reduce qubit counts or shots if your computer runs low on memory or time.
